# Lab 8: Two Ways to Generate

The question I'm working on: on the same small digits, how do a GAN and a diffusion model compare, and why does one kind of GAN fall apart where the other doesn't? I write the GAN losses, WGAN-GP and a DDPM diffusion model from scratch, check each one against something I can work out exactly, and then score them next to the starter's VAE. The starter supplies the data, the networks, the noise schedule, the VAE and the scoring, unchanged. Every trained model gets seeds 0, 1 and 2.

In [1]:
import json
import math
import os
import platform
import subprocess
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import lab08_starter as S

SEEDS = [0, 1, 2]
SAMPLE_SEED = 9            # noise seed for every evaluation draw (matches S.VAE.sample's default)
N_SAMPLES = 359            # every Frechet distance, precision and recall uses 359 images
DEVICE = "cpu"
BATCH_SIZE = 128
GAN_STEPS = 4000           # generator steps
GAN_BETAS = (0.5, 0.999)
STANDARD_LR_G, STANDARD_LR_D = 5e-4, 2e-4
WGAN_LR, WGAN_CRITIC_STEPS, GP_WEIGHT = 5e-4, 3, 10.0
DIFFUSION_STEPS = 4000
RING_HIDDEN, RING_LR = 128, 2e-3
DIGIT_HIDDEN, DIGIT_LR = 256, 1e-3
T_STEPS = 100
DETERMINISTIC_KS = [5, 10, 25, 50, 100]
os.makedirs("figures", exist_ok=True)
results = {}
run_start = time.perf_counter()

print("python", platform.python_version(), "| torch", torch.__version__, "| threads", torch.get_num_threads(),
      "| venv kernel:", sys.prefix.endswith(".venv"))

python 3.12.13 | torch 2.2.2 | threads 8 | venv kernel: True


The imports and settings are in place. Every run below uses seeds 0, 1 and 2, and every evaluation draw uses sample seed 9, so any two runs I compare start from the same noise.

## Run the starter once

Before I change anything I want to know the environment works, so I run the starter unmodified.

In [2]:
starter_start = time.perf_counter()
starter_run = subprocess.run([sys.executable, "lab08_starter.py"], capture_output=True, text=True, check=True)
print(starter_run.stdout)
print(f"starter took {time.perf_counter() - starter_start:.1f}s")

train (1438, 64) test (359, 64)
modes covered by real ring data: 8
alpha_bar at the last step: 0.005618760362267494
baseline VAE (30 epochs): {'frechet': 31.143, 'precision': 0.847, 'recall': 0.201, 'nearest_train': 1.269}
saved starter_check.png

starter took 5.7s


The starter runs as given. Its 30-epoch VAE scores Fréchet 31.1, precision 0.85 and recall 0.20, which already tells me something: on this feature network a weak generator can look precise and still cover very little of the data.

## One ruler for every score

Comparing generators only means something if they're all measured the same way. So I train the starter's feature network once (seed 0) and use it for every Fréchet distance, precision and recall in this notebook, always 359 generated images against the 359 test images. On top of the starter's scores I add:

- **Nearest train:** how close each sample is to a training image, to catch copying.
- **Digit counts:** which digit the classifier thinks each sample is, to catch dropped modes.
- **Digit TV:** how far the digit mix is from the test mix.

The scorer refuses anything outside [0, 1], so a diffusion sample can't be scored before it's rescaled.

In [3]:
Xtr, ytr, Xte, yte = S.load_task()
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))
print(f"pixel range: {Xtr.min().item():.2f} to {Xtr.max().item():.2f}")
print("train images per digit:", torch.bincount(ytr, minlength=10).tolist())
print("test images per digit: ", torch.bincount(yte, minlength=10).tolist())

feature_net = S.train_feature_net(Xtr, ytr)       # seed 0 inside; trained ONCE, reused for every score
test_features = S.get_features(feature_net, Xte)
with torch.no_grad():
    feature_net_accuracy = (feature_net(Xte).argmax(1) == yte).float().mean().item()
test_shares = torch.bincount(yte, minlength=10).double() / len(yte)
print(f"feature network test accuracy: {feature_net_accuracy:.3f}")


def score_images(images, expected_count=N_SAMPLES):
    """Score images that are ALREADY in [0, 1] (diffusion samples must go through to_unit_range first).
    Returns the starter's four scores plus nearest_train_min, digit_counts, digits_present, tv."""
    assert len(images) == expected_count, f"expected {expected_count} images, got {len(images)}"
    assert images.min() >= 0 and images.max() <= 1, "images must be in [0, 1] before scoring"
    images = images.detach()
    scores = S.score_samples(images, test_features, feature_net, Xtr)
    distances = S.nearest_train_distance(images, Xtr)
    with torch.no_grad():
        predicted = feature_net(images).argmax(1)
    counts = torch.bincount(predicted, minlength=10)
    shares = counts.double() / len(images)
    scores.update({"nearest_train_min": float(distances.min()),
                   "digit_counts": [int(c) for c in counts],
                   "digits_present": int((counts > 0).sum()),
                   "tv": float(0.5 * (shares - test_shares).abs().sum())})
    return scores


# Reference block: what the scores look like for real images
test_to_train = S.nearest_train_distance(Xte, Xtr)
subset_generator = torch.Generator().manual_seed(0)
train_subset = Xtr[torch.randperm(len(Xtr), generator=subset_generator)[:N_SAMPLES]]
results["feature_net_accuracy"] = feature_net_accuracy
results["reference"] = {
    "test_nearest_train_mean": float(test_to_train.mean()),
    "test_nearest_train_min": float(test_to_train.min()),
    "test_digit_counts": [int(c) for c in torch.bincount(yte, minlength=10)],
    "train_subset_score": score_images(train_subset),   # nearest-train is 0 here: these images are training images
}
print("reference: real test images to nearest training image, mean "
      f"{results['reference']['test_nearest_train_mean']:.3f}, min {results['reference']['test_nearest_train_min']:.3f}")
print("reference: 359 real training images scored as if generated:",
      {k: (round(v, 3) if isinstance(v, float) else v) for k, v in results["reference"]["train_subset_score"].items()})

train (1438, 64) test (359, 64)
pixel range: 0.00 to 1.00
train images per digit: [151, 147, 141, 154, 151, 142, 137, 140, 136, 139]
test images per digit:  [27, 35, 36, 29, 30, 40, 44, 39, 38, 41]


feature network test accuracy: 0.972
reference: real test images to nearest training image, mean 1.068, min 0.538
reference: 359 real training images scored as if generated: {'frechet': 2.833, 'precision': 0.9, 'recall': 0.911, 'nearest_train': 0.0, 'nearest_train_min': 0.0, 'digit_counts': [43, 39, 27, 38, 35, 34, 39, 45, 33, 26], 'digits_present': 10, 'tv': 0.111}


The feature network gets 0.972 on the test digits. To know what a perfect generator would score, I scored 359 real training images as if they were generated:

- **Fréchet 2.83, precision 0.90, recall 0.91:** so even perfect samples don't reach recall 1 with 359 images.
- **Nearest train:** real test images sit 1.07 from their nearest training image on average and 0.54 at the closest. That's what a genuinely new digit looks like; a copy would be near 0.

## Piece 1: GAN losses from the logit

The discriminator gives a logit `s`, and D = sigmoid(s). Taking the log of a sigmoid that has saturated at 0 or 1 gives -inf, so I write every loss with softplus instead:

- **Discriminator:** -mean log D(real) - mean log(1 - D(fake)).
- **Original generator loss:** mean log(1 - D(G(z))), minimised.
- **Non-saturating generator loss:** -mean log D(G(z)).

I don't trust them until they pass three checks.

In [4]:
def _reduce(values, reduction):
    assert reduction in ("mean", "sum")
    return values.mean() if reduction == "mean" else values.sum()


def discriminator_loss(real_logits, fake_logits, reduction="mean"):
    """-log D(real) - log(1 - D(fake)) with D = sigmoid(logit). Uses log sigmoid(s) = -softplus(-s) and log(1 - sigmoid(s)) = -softplus(s)."""
    return _reduce(F.softplus(-real_logits), reduction) + _reduce(F.softplus(fake_logits), reduction)


def generator_loss_original(fake_logits, reduction="mean"):
    """The original generator loss, log(1 - D(G(z))). The generator minimizes this one."""
    return _reduce(-F.softplus(fake_logits), reduction)


def generator_loss_nonsaturating(fake_logits, reduction="mean"):
    """The non-saturating generator loss, -log D(G(z))."""
    return _reduce(F.softplus(-fake_logits), reduction)

### Check 1: each loss against its definition

I compare each loss with the textbook formula on 1,000 random logits in float64, scaled up so many are far from zero. Then I try logits of -40, 0 and 40 in float32, where the naive formula should break.

In [5]:
torch.manual_seed(0)
logits_real = torch.randn(1000, dtype=torch.float64) * 4
logits_fake = torch.randn(1000, dtype=torch.float64) * 4
naive_d = -(torch.log(torch.sigmoid(logits_real)).mean() + torch.log(1 - torch.sigmoid(logits_fake)).mean())
naive_gmm = torch.log(1 - torch.sigmoid(logits_fake)).mean()
naive_gns = -torch.log(torch.sigmoid(logits_fake)).mean()
loss_rows = [("discriminator loss", discriminator_loss(logits_real, logits_fake), naive_d),
             ("generator loss, original", generator_loss_original(logits_fake), naive_gmm),
             ("generator loss, non-saturating", generator_loss_nonsaturating(logits_fake), naive_gns)]
check_losses = {"rows": {}}
print(f"{'loss':32s} {'mine':>20s} {'definition':>20s} {'abs diff':>10s}")
for name, mine, definition in loss_rows:
    diff = float((mine - definition).abs())
    check_losses["rows"][name] = {"mine": float(mine), "definition": float(definition), "abs_diff": diff}
    print(f"{name:32s} {float(mine):20.12f} {float(definition):20.12f} {diff:10.2e}")
check_losses["max_abs_diff"] = max(row["abs_diff"] for row in check_losses["rows"].values())
check_losses["pass"] = check_losses["max_abs_diff"] < 1e-10
print("max abs diff", f"{check_losses['max_abs_diff']:.2e}", "| pass (< 1e-10):", check_losses["pass"])

stability_logits = torch.tensor([-40.0, 0.0, 40.0])
naive_stability = torch.log(1 - torch.sigmoid(stability_logits))
mine_stability = -F.softplus(stability_logits)
check_losses["stability_float32"] = {"logits": stability_logits.tolist(), "naive_log_one_minus_sigmoid": naive_stability.tolist(),
                                     "mine": mine_stability.tolist()}
print("float32 stability, logits", stability_logits.tolist())
print("  naive log(1 - sigmoid(s)):", naive_stability.tolist())
print("  mine  -softplus(s):       ", mine_stability.tolist())
results["check_losses"] = check_losses

loss                                             mine           definition   abs diff
discriminator loss                     3.369809185009       3.369809185009   2.80e-14
generator loss, original              -1.644090479101      -1.644090479101   2.80e-14
generator loss, non-saturating         1.814742238089       1.814742238089   0.00e+00
max abs diff 2.80e-14 | pass (< 1e-10): True
float32 stability, logits [-40.0, 0.0, 40.0]
  naive log(1 - sigmoid(s)): [0.0, -0.6931471824645996, -inf]
  mine  -softplus(s):        [-4.24835413113866e-18, -0.6931471824645996, -40.0]


All three match the definitions to about 1e-14. The float32 row shows why softplus matters: at logit 40 the naive log(1 - sigmoid) is -inf and mine is -40. That's exactly the region a winning discriminator pushes into.

### Check 2: does the discriminator learn the optimal one?

For two known distributions the best possible discriminator is p_data / (p_data + p_g). Real data is N(0, 1) and a fixed generator is N(1.5, 1), so I train a small 1-D discriminator with my loss and compare it with that formula from -2 to 3.5.

In [6]:
def optimal_discriminator_gap(seed):
    """Train a 1-D discriminator on N(0,1) against a fixed N(1.5,1) generator; return the largest gap to the optimal D and where it is."""
    torch.manual_seed(seed)
    disc = S.Discriminator(d_in=1, d_hidden=32)
    optimizer = torch.optim.Adam(disc.parameters(), lr=3e-3)           # default betas (0.9, 0.999), stated
    for _ in range(3000):
        real = torch.randn(256, 1)
        fake = torch.randn(256, 1) + 1.5                                # fixed generator N(1.5, 1)
        loss = discriminator_loss(disc(real), disc(fake))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    grid = torch.linspace(-2.0, 3.5, 551)                               # step 0.01, endpoints included
    p_data = torch.exp(-grid ** 2 / 2) / math.sqrt(2 * math.pi)
    p_gen = torch.exp(-(grid - 1.5) ** 2 / 2) / math.sqrt(2 * math.pi)
    optimal = p_data / (p_data + p_gen)
    with torch.no_grad():
        learned = torch.sigmoid(disc(grid[:, None]))
    gaps = (learned - optimal).abs()
    return float(gaps.max()), float(grid[gaps.argmax()])


check_optimal_discriminator = {}
print(f"{'seed':>4s} {'largest gap':>12s} {'at x':>7s}")
for seed in SEEDS:
    gap, location = optimal_discriminator_gap(seed)
    check_optimal_discriminator[str(seed)] = {"largest_gap": gap, "at_x": location, "pass": gap < 0.05}
    print(f"{seed:4d} {gap:12.5f} {location:7.2f}   pass (< 0.05): {gap < 0.05}")
results["check_optimal_discriminator"] = check_optimal_discriminator

seed  largest gap    at x


   0      0.01705    1.48   pass (< 0.05): True


   1      0.03187    0.32   pass (< 0.05): True


   2      0.02806    1.23   pass (< 0.05): True


The trained discriminator stays within 0.017, 0.032 and 0.028 of the optimal one on the three seeds, all under the 0.05 bound. The biggest gaps are small wiggles in the middle, not a wrong trend, so my discriminator loss finds the right answer.

### Check 3: does the original generator loss saturate?

The module says the original generator loss stops teaching once the discriminator is winning. To test that I shift a generator's output 6 units away from the ring data, train a discriminator for 300 steps so it easily rejects those samples, and then measure the gradient each generator loss sends back to 1,000 generated points.

In [7]:
ring_points, _ = S.ring_of_gaussians(5000)          # data seed 0 (starter default)


def saturation_check(seed):
    """Shift a fresh generator's output by 6, train D for 300 steps, then measure the gradient each generator loss sends to the output."""
    torch.manual_seed(seed)
    generator = S.Generator(d_noise=8, d_out=2)                         # hidden 64, starter default
    with torch.no_grad():
        generator.net[-1].bias += torch.tensor([6.0, 6.0])              # shift the output by 6 in both coordinates
    disc = S.Discriminator()                                            # d_in=2, hidden 64
    optimizer = torch.optim.Adam(disc.parameters(), lr=2e-3, betas=(0.5, 0.9))
    for _ in range(300):
        real = ring_points[torch.randint(0, len(ring_points), (128,))]
        with torch.no_grad():
            fake = generator(torch.randn(128, 8))
        loss = discriminator_loss(disc(real), disc(fake))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    with torch.no_grad():
        generated = generator(torch.randn(1000, 8))
    generated = generated.clone().requires_grad_(True)                  # leaf: "the generator's output"
    logits = disc(generated)
    out = {"mean_d_of_g": float(torch.sigmoid(logits).mean())}
    for name, loss_fn in [("original", generator_loss_original), ("nonsaturating", generator_loss_nonsaturating)]:
        grad, = torch.autograd.grad(loss_fn(logits, reduction="sum"), generated, retain_graph=True)
        out[f"grad_norm_{name}"] = float(grad.norm(dim=1).mean())      # per-sample norm, then mean
    return out


check_saturation = {}
print(f"{'seed':>4s} {'mean D(G(z))':>14s} {'orig grad norm':>16s} {'nonsat grad norm':>18s} {'ratio':>10s}")
for seed in SEEDS:
    row = saturation_check(seed)
    row["ratio_nonsat_over_orig"] = row["grad_norm_nonsaturating"] / row["grad_norm_original"]
    row["pass"] = bool(row["mean_d_of_g"] < 0.01 and row["grad_norm_original"] < 1e-2
                       and row["grad_norm_nonsaturating"] > 0.1 and row["ratio_nonsat_over_orig"] > 100)
    check_saturation[str(seed)] = row
    print(f"{seed:4d} {row['mean_d_of_g']:14.5f} {row['grad_norm_original']:16.5f} {row['grad_norm_nonsaturating']:18.5f} {row['ratio_nonsat_over_orig']:10.1f}"
          f"   | D(G(z)) {row['mean_d_of_g']:.3e}, orig {row['grad_norm_original']:.3e}, nonsat {row['grad_norm_nonsaturating']:.3e}   pass: {row['pass']}")
results["check_saturation"] = check_saturation

seed   mean D(G(z))   orig grad norm   nonsat grad norm      ratio


   0        0.00000          0.00000            5.84570 641827793.7   | D(G(z)) 1.561e-09, orig 9.108e-09, nonsat 5.846e+00   pass: True


   1        0.00000          0.00000            5.77351 1073740139.1   | D(G(z)) 9.255e-10, orig 5.377e-09, nonsat 5.774e+00   pass: True


   2        0.00000          0.00000            5.66609 684134310.6   | D(G(z)) 1.459e-09, orig 8.282e-09, nonsat 5.666e+00   pass: True


It does saturate. The discriminator gives the generated points a probability of about 1e-9, and then:

- **Original loss:** the gradient per point is about 1e-8, basically nothing.
- **Non-saturating loss:** the gradient is about 5.7 to 5.8.

That's a ratio of roughly 1e9 on every seed. The original loss stops teaching exactly when the generator needs it most, which is why the GANs below use the non-saturating one.